# 🚀 SAM-AI R1: High-Velocity GRPO Reinforcement Learning

**Pillar 1:** Verifiable System 2 Thinking Calibration (Math & ARC-AGI)
**Target Model:** DeepSeek-R1 Distill Architecture
**Optimization:** Group Relative Policy Optimization (GRPO) + RLVR
**Efficiency:** 500 Curated Tasks, 1 Concentrated Epoch, Hub Checkpoint Streaming
**Target Wall-Clock Runtime:** 60 - 90 Minutes


In [ ]:
# 1. Environment & Dependency Conflict Resolution
import subprocess, sys, os
print('[*] Resolving Kaggle pre-installed environment conflicts...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
pkgs = ['trl>=0.15.0', 'peft>=0.12.0', 'transformers>=4.48.0', 'accelerate>=1.2.0', 'datasets', 'bitsandbytes', 'huggingface_hub']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('[✓] Modern TRL, PEFT, and Transformers ready.')


In [ ]:
# 2. Hardware Accelerator Diagnostic
import torch
print('=' * 70)
print('  SAM-AI HIGH-VELOCITY HARDWARE DIAGNOSTIC')
print('=' * 70)
if not torch.cuda.is_available():
    print('[!] CUDA not detected. Ensure GPU accelerator is enabled in Kaggle settings.')
    gpu_name = 'CPU'; vram_gb = 0.0; bf16_supported = False
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    bf16_supported = torch.cuda.is_bf16_supported()
    print(f'[✓] GPU Accelerator: {gpu_name} ({vram_gb:.2f} GB VRAM)')
    print(f'[✓] bfloat16 Acceleration: {bf16_supported}')
print('=' * 70)


In [ ]:
# 3. Model Architecture & Hyperparameter Allocation
if vram_gb >= 35.0:
    MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-14B'
    LOAD_4BIT = False
    BATCH_SIZE = 2
else:
    # 1.5B fits in 3GB VRAM natively on T4, leaving 11GB for GRPO rollouts
    MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
    LOAD_4BIT = False
    BATCH_SIZE = 2

GROUP_SIZE = 4
MAX_COMPLETION_LENGTH = 384
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 5e-6
NUM_TRAIN_EPOCHS = 1
OUTPUT_DIR = '/kaggle/working/sam_ai_r1_fast_trained'
HF_REPO = 'Samrish2009/SAM-AI-Reasoning-14B'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
print(f'[*] Provisioned Model: {MODEL_ID} (4-Bit: {LOAD_4BIT})')
print(f'[*] Reasoning Budget: {MAX_COMPLETION_LENGTH} tokens per completion')
print(f'[*] Parallel Rollouts: {GROUP_SIZE} generations per prompt')


In [ ]:
# 4. Curated 500-Sample High-Signal Dataset (250 ARC + 250 Math)
import random
from datasets import Dataset

def format_grid(grid):
    return '\n'.join(' '.join(str(c) for c in row) for row in grid)

def rotate_90(grid):
    return [list(row) for row in zip(*grid[::-1])]

def reflect_h(grid):
    return [row[::-1] for row in grid]

def reflect_v(grid):
    return grid[::-1]

def color_remap(grid, old_c, new_c):
    return [[new_c if c == old_c else c for c in row] for row in grid]

def generate_arc_dataset(n_samples=250):
    random.seed(42)
    tasks = []
    for _ in range(n_samples):
        h, w = random.randint(3, 4), random.randint(3, 4)
        colors = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
        used = random.sample(colors, random.randint(2, 3))
        grid = [[random.choice(used) for _ in range(w)] for _ in range(h)]
        rule = random.choice(['rot90', 'rot180', 'flip_h', 'flip_v', 'color_shift'])
        if rule == 'rot90':
            out = rotate_90(grid); desc = 'Rotate grid 90 degrees clockwise.'
        elif rule == 'rot180':
            out = rotate_90(rotate_90(grid)); desc = 'Rotate grid 180 degrees.'
        elif rule == 'flip_h':
            out = reflect_h(grid); desc = 'Reflect grid horizontally (left-right flip).'
        elif rule == 'flip_v':
            out = reflect_v(grid); desc = 'Reflect grid vertically (upside-down flip).'
        else:
            c1, c2 = random.sample(used, 2)
            out = color_remap(grid, c1, c2); desc = f'Replace color {c1} with {c2}.'
        prompt = (
            f'[ARC-AGI Task] {desc}\n\nInput Grid ({h}x{w}):\n{format_grid(grid)}\n\n'
            f'Derive output step-by-step in <think>...</think>. Output transformed grid inside <answer>...</answer>.'
        )
        tasks.append({'prompt': prompt, 'answer': format_grid(out), 'domain': 'arc_spatial'})
    return tasks

def generate_math_dataset(n_samples=250):
    random.seed(101)
    tasks = []
    for _ in range(n_samples):
        kind = random.choice(['arithmetic', 'algebra', 'modulo'])
        if kind == 'arithmetic':
            a, b, c = random.randint(15, 60), random.randint(10, 40), random.randint(3, 8)
            ans = (a + b) * c
            prompt = f'Evaluate step-by-step using <think> tags: ({a} + {b}) * {c} = ? Final answer inside <answer>.'
        elif kind == 'algebra':
            x, m, k = random.randint(3, 25), random.randint(3, 7), random.randint(10, 40)
            rhs = m * x + k; ans = x
            prompt = f'Solve for x step-by-step: {m}x + {k} = {rhs}. Put integer value of x inside <answer>.'
        else:
            val, mod = random.randint(100, 750), random.choice([7, 11, 13])
            ans = val % mod
            prompt = f'Calculate remainder of {val} divided by {mod} using <think> tags. Put integer inside <answer>.'
        tasks.append({'prompt': prompt, 'answer': str(ans), 'domain': 'verifiable_math'})
    return tasks

print('[*] Generating high-signal curated tasks...')
arc_data = generate_arc_dataset(250)
math_data = generate_math_dataset(250)
all_data = arc_data + math_data
random.seed(777)
random.shuffle(all_data)
train_dataset = Dataset.from_list([
    {
        'prompt': [
            {'role': 'system', 'content': 'You are SAM-AI, a sovereign reasoning intelligence. Solve with rigorous System 2 deliberation inside <think>...</think> and place final answer in <answer>...</answer>.'},
            {'role': 'user', 'content': p['prompt']}
        ],
        'answer': p['answer'],
        'domain': p['domain']
    }
    for p in all_data
])
print(f'[✓] High-signal dataset ready: {len(train_dataset)} tasks (250 ARC + 250 Math).')


In [ ]:
# 5. Deterministic RLVR Verifier
import re
def extract_answer_tag(text):
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else None

def sam_rlvr_reward_fn(completions, answer=None, domain=None, **kwargs):
    rewards = []
    n_comp = len(completions)
    ans_list = answer if answer is not None else []
    dom_list = domain if domain is not None else []
    if len(ans_list) > 0 and len(ans_list) != n_comp:
        rep = max(1, n_comp // len(ans_list))
        ans_list = [a for a in ans_list for _ in range(rep)][:n_comp]
        dom_list = [d for d in dom_list for _ in range(rep)][:n_comp]
    elif len(ans_list) == 0:
        ans_list = [''] * n_comp; dom_list = [''] * n_comp

    for i, completion in enumerate(completions):
        target = ans_list[i] if i < len(ans_list) else ''
        d = dom_list[i] if i < len(dom_list) else ''
        text = completion[-1]['content'] if isinstance(completion, list) and completion and isinstance(completion[-1], dict) else str(completion)
        r = 0.0
        # Thinking format reward
        if '<think>' in text and '</think>' in text:
            tc = text.split('</think>')[0].replace('<think>', '').strip()
            if len(tc) >= 20: r += 0.35
        if '<answer>' in text and '</answer>' in text: r += 0.15
        extracted = extract_answer_tag(text)
        target_clean = str(target).strip()
        if d == 'arc_spatial':
            norm_target = re.sub(r'\s+', ' ', target_clean)
            if extracted:
                norm_pred = re.sub(r'\s+', ' ', extracted.strip())
                if norm_pred == norm_target: r += 1.0
                elif len(norm_pred) > 0 and (norm_pred in norm_target or norm_target in norm_pred): r += 0.35
            elif norm_target in re.sub(r'\s+', ' ', text): r += 0.5
        else:
            if extracted and extracted == target_clean: r += 1.0
            elif target_clean in re.findall(r'\b\d+\b', text): r += 0.5
        rewards.append(r)
    return rewards
print('[✓] Deterministic RLVR verifier registered.')


In [ ]:
# 6. Model Initialization & LoRA Injection
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
print(f'[*] Initializing tokenizer: {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16
if LOAD_4BIT:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={'': 0}, trust_remote_code=True)
else:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=compute_dtype, device_map={'': 0}, trust_remote_code=True)
lora_config = LoraConfig(r=16, lora_alpha=16, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(base_model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'[✓] LoRA injected: {trainable:,} trainable / {total:,} total params ({100 * trainable / total:.2f}%).')


In [ ]:
# 7. High-Velocity GRPO Training with Live Hugging Face Checkpointing
import inspect
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback

class HubPushCallback(TrainerCallback):
    '''Automatically pushes checkpoint to Hugging Face Hub on save so zero work is ever lost.'''
    def on_save(self, args, state, control, **kwargs):
        try:
            step = state.global_step
            print(f'\n[🔄] Checkpoint step {step}: Streaming weights live to {HF_REPO}...')
            model.push_to_hub(HF_REPO, token=HF_TOKEN, commit_message=f'SAM-AI R1 checkpoint step {step}')
            tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
            print(f'[✓] Step {step} safely preserved on Hugging Face Hub!')
        except Exception as e:
            print(f'[Notice] Intermediate Hub sync notice: {e}')

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=GROUP_SIZE,
    max_completion_length=MAX_COMPLETION_LENGTH,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    save_steps=25,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    report_to='none',
)

grpo_init_kwargs = {}
sig = inspect.signature(GRPOTrainer.__init__)
if 'processing_class' in sig.parameters:
    grpo_init_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in sig.parameters:
    grpo_init_kwargs['tokenizer'] = tokenizer

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=sam_rlvr_reward_fn,
    callbacks=[HubPushCallback()],
    **grpo_init_kwargs
)
print('=' * 70)
print('  LAUNCHING HIGH-VELOCITY GRPO REINFORCEMENT LEARNING RUN')
print('=' * 70)
trainer.train()
print('[*] Final training complete. Packaging final artifacts...')
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
try:
    model.push_to_hub(HF_REPO, token=HF_TOKEN, commit_message='SAM-AI R1 Final Trained Weights')
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f'[🎉] SAM-AI R1 IS OFFICIALLY LIVE: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'[Notice] Final Hub sync notice: {e}')


In [ ]:
# 8. Held-Out System 2 Verification Demonstration
test_sample = arc_data[0]['prompt']
print('[*] Verifying SAM-AI R1 System 2 Thinking on Held-Out ARC Spatial Grid:\n')
print(test_sample)
inputs = tokenizer(
    f'<|im_start|>system\nYou are SAM-AI, a sovereign reasoning intelligence. Think step-by-step using <think> tags and output final answer in <answer> tags.<|im_end|>\n<|im_start|>user\n{test_sample}<|im_end|>\n<|im_start|>assistant\n',
    return_tensors='pt'
).to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=384, temperature=0.6, top_p=0.95, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('=' * 70)
print('  SAM-AI R1 REASONING TRACE & GENERATED SOLUTION')
print('=' * 70)
print(response)
print('=' * 70)
